# 02 — Demo: Lakebase branching lifecycle

Walks through what the app's `Onboard new advertiser` button does under the hood, so you can show it from a notebook too:

1. List current branches off `maas-team8`
2. Create a new branch (POST /api/2.0/database/instances with `parent_instance_ref`)
3. Connect to the branch — verify the synced data is there
4. Tear it down

End-to-end takes ~30 seconds. Great for the readout if the UI is acting up.

In [ ]:
import os, uuid, json, time, asyncio
from datetime import datetime, timezone
from databricks.sdk import WorkspaceClient
import asyncpg

INSTANCE = os.environ.get('LAKEBASE_INSTANCE', 'maas-team8')
LB_DB    = os.environ.get('LAKEBASE_DATABASE', 'acme_agency')
BRANCH   = f'maas-team8-demo-{int(time.time()) % 100000}'

w = WorkspaceClient()
print(f'parent={INSTANCE}  new branch={BRANCH}')

In [ ]:
# 1. List branches that already exist
r = w.api_client.do('GET', '/api/2.0/database/instances')
branches = [i for i in r.get('database_instances', [])
             if (i.get('parent_instance_ref') or {}).get('name') == INSTANCE]
print(f'{len(branches)} existing branch(es) off {INSTANCE}')
for b in branches:
    print(f"  {b['name']:55s} state={b.get('state')}")

In [ ]:
# 2. Spawn a branch
branch_time = datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')
body = {
    'name': BRANCH,
    'capacity': 'CU_1',
    'parent_instance_ref': {'name': INSTANCE, 'branch_time': branch_time},
}
resp = w.api_client.do('POST', '/api/2.0/database/instances', body=body)
print(json.dumps({k: resp.get(k) for k in ('name','state','uid','read_write_dns')}, indent=2))

while resp.get('state') not in ('AVAILABLE','FAILED'):
    time.sleep(8)
    resp = w.api_client.do('GET', f'/api/2.0/database/instances/{BRANCH}')
    print(f"  state={resp.get('state')}")
print(f'branch ready: {BRANCH}')

In [ ]:
# 3. Connect to the branch and verify the synced data is queryable on the copy
me = w.current_user.me().user_name
host = resp['read_write_dns']
cred = w.api_client.do('POST', '/api/2.0/database/credentials',
                       body={'instance_names':[BRANCH], 'request_id': str(uuid.uuid4())})

async def go():
    conn = await asyncpg.connect(host=host, port=5432, user=me, password=cred['token'],
                                  database=LB_DB, ssl='require')
    n_adv  = await conn.fetchval('SELECT COUNT(*) FROM public.advertisers')
    n_perf = await conn.fetchval('SELECT COUNT(*) FROM public.campaign_daily_perf')
    pat    = await conn.fetchval('SELECT SUM(spend_usd) FROM public.campaign_daily_perf WHERE advertiser_id=$1','adv_patagonia')
    print(f'branch sees: {n_adv} advertisers, {n_perf:,} perf rows, patagonia 12w spend ${pat:,.0f}')
    await conn.close()
asyncio.run(go())

In [ ]:
# 4. Tear it down
w.api_client.do('DELETE', f'/api/2.0/database/instances/{BRANCH}')
print(f'deleted {BRANCH}')